In [1]:
import math
import os
import sys
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Callable

import datasets
import einops
import numpy as np
import torch as t
import torch.nn as nn
import wandb
from jaxtyping import Float, Int
from rich import print as rprint
from rich.table import Table
from torch import Tensor
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformer_lens import HookedTransformer
from transformer_lens.utils import gelu_new, tokenize_and_concatenate
from transformers import GPT2TokenizerFast

device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")

MAIN = __name__ == "__main__"

# Make sure exercises are in the path
chapter = "chapter1_transformer_interp"
section = "part1_transformer_from_scratch"
root_dir = next(p for p in Path.cwd().parents if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

/tmp/ipykernel_2424/373356113.py:22: DeprecationWarning: The 'utils' module has been deprecated. Please use 'transformer_lens.utilities' instead. Importing from utils.py will be removed in TransformerLens 4.0.
  from transformer_lens.utils import gelu_new, tokenize_and_concatenate


In [2]:
# Disabling HTML Tqdm bars
import os
os.environ["TQDM_DISABLE"] = "1" # or use text mode

# Load the model
import part1_transformer_from_scratch.solutions as solutions
import part1_transformer_from_scratch.tests as tests
reference_gpt2 = HookedTransformer.from_pretrained(
    "gpt2-small",
    fold_ln=False,
    center_unembed=False,
    center_writing_weights=False,  # you'll learn about these arguments later!
    device=device,
)

/tmp/ipykernel_2424/3941548315.py:8: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  reference_gpt2 = HookedTransformer.from_pretrained(


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2-small into HookedTransformer


In [3]:
reference_text = "The cat sat on the"
tokens = reference_gpt2.to_tokens(reference_text).to(device)
print(f'tokens.shape:\t\t\t {tokens.shape}')
print(f'tokens:\t\t\t\t {tokens}')
print(f'gpt2.to_str_tokens(tokens):\t {reference_gpt2.to_str_tokens(tokens)}')

logits, cache = reference_gpt2.run_with_cache(tokens)
print(f'logits.shape:\t\t\t {logits.shape}')
probs = logits.softmax(dim=-1)
arg_max = logits.argmax(dim=-1)
print(f'logits.argmax(dim=-1):\t\t {arg_max}')

# Max token index for each input token
print(f'arg_max[0].unsqueeze(-1).shape:\t {arg_max[0].unsqueeze(-1).shape}')

most_likely_next_tokens = reference_gpt2.to_str_tokens(arg_max[0])
# most_likely_next_tokens = reference_gpt2.tokenizer.batch_decode(logits.argmax(dim=-1)[0].unsqueeze(-1))
print('\n(Current Token, Most Likely Next Token)')
print(list(zip(reference_gpt2.to_str_tokens(tokens), most_likely_next_tokens)))

tokens.shape:			 torch.Size([1, 6])
tokens:				 tensor([[50256,   464,  3797,  3332,   319,   262]], device='cuda:0')
gpt2.to_str_tokens(tokens):	 ['<|endoftext|>', 'The', ' cat', ' sat', ' on', ' the']
logits.shape:			 torch.Size([1, 6, 50257])
logits.argmax(dim=-1):		 tensor([[ 198,  717,  373,  319,  262, 4314]], device='cuda:0')
arg_max[0].unsqueeze(-1).shape:	 torch.Size([6, 1])

(Current Token, Most Likely Next Token)
[('<|endoftext|>', '\n'), ('The', ' first'), (' cat', ' was'), (' sat', ' on'), (' on', ' the'), (' the', ' floor')]


In [4]:
print(f'All Activation Shapes of Reference Model')
for activation_name, activation in cache.items():
    # Only print for first layer
    if ".0." in activation_name or "blocks" not in activation_name:
        print(f"{activation_name:30} {tuple(activation.shape)}")

All Activation Shapes of Reference Model
hook_embed                     (1, 6, 768)
hook_pos_embed                 (1, 6, 768)
blocks.0.hook_resid_pre        (1, 6, 768)
blocks.0.ln1.hook_scale        (1, 6, 1)
blocks.0.ln1.hook_normalized   (1, 6, 768)
blocks.0.attn.hook_q           (1, 6, 12, 64)
blocks.0.attn.hook_k           (1, 6, 12, 64)
blocks.0.attn.hook_v           (1, 6, 12, 64)
blocks.0.attn.hook_attn_scores (1, 12, 6, 6)
blocks.0.attn.hook_pattern     (1, 12, 6, 6)
blocks.0.attn.hook_z           (1, 6, 12, 64)
blocks.0.hook_attn_out         (1, 6, 768)
blocks.0.hook_resid_mid        (1, 6, 768)
blocks.0.ln2.hook_scale        (1, 6, 1)
blocks.0.ln2.hook_normalized   (1, 6, 768)
blocks.0.mlp.hook_pre          (1, 6, 3072)
blocks.0.mlp.hook_post         (1, 6, 3072)
blocks.0.hook_mlp_out          (1, 6, 768)
blocks.0.hook_resid_post       (1, 6, 768)
ln_final.hook_scale            (1, 6, 1)
ln_final.hook_normalized       (1, 6, 768)
unembed.hook_in                (1, 6, 768)
u

In [5]:
print("All Parameter Shapes of Reference Model")
for name, param in reference_gpt2.named_parameters():
    # Only print for first layer
    if ".0." in name or "blocks" not in name:
        print(f"{name:18} {tuple(param.shape)}")

All Parameter Shapes of Reference Model
embed.W_E          (50257, 768)
pos_embed.W_pos    (1024, 768)
blocks.0.ln1.w     (768,)
blocks.0.ln1.b     (768,)
blocks.0.ln2.w     (768,)
blocks.0.ln2.b     (768,)
blocks.0.attn.W_Q  (12, 768, 64)
blocks.0.attn.W_O  (12, 64, 768)
blocks.0.attn.b_Q  (12, 64)
blocks.0.attn.b_O  (768,)
blocks.0.attn.W_K  (12, 768, 64)
blocks.0.attn.W_V  (12, 768, 64)
blocks.0.attn.b_K  (12, 64)
blocks.0.attn.b_V  (12, 64)
blocks.0.mlp.W_in  (768, 3072)
blocks.0.mlp.b_in  (3072,)
blocks.0.mlp.W_out (3072, 768)
blocks.0.mlp.b_out (768,)
ln_final.w         (768,)
ln_final.b         (768,)
unembed.W_U        (768, 50257)
unembed.b_U        (50257,)


In [6]:
# As a reference - note there's a lot of stuff we don't care about in here, to do with library internals or other architectures
print(reference_gpt2.cfg)

HookedTransformerConfig:
{'NTK_by_parts_factor': 8.0,
 'NTK_by_parts_high_freq_factor': 4.0,
 'NTK_by_parts_low_freq_factor': 1.0,
 'NTK_original_ctx_len': 8192,
 'act_fn': 'gelu_new',
 'attention_dir': 'causal',
 'attn_only': False,
 'attn_scale': np.float64(8.0),
 'attn_scores_soft_cap': -1.0,
 'attn_types': None,
 'checkpoint_index': None,
 'checkpoint_label_type': None,
 'checkpoint_value': None,
 'clip_qkv': None,
 'd_head': 64,
 'd_mlp': 3072,
 'd_model': 768,
 'd_vocab': 50257,
 'd_vocab_out': 50257,
 'decoder_start_token_id': None,
 'default_prepend_bos': True,
 'device': device(type='cuda'),
 'dtype': torch.float32,
 'eps': 1e-05,
 'experts_per_token': None,
 'final_rms': False,
 'from_checkpoint': False,
 'gated_mlp': False,
 'init_mode': 'gpt2',
 'init_weights': False,
 'initializer_range': np.float64(0.02886751345948129),
 'layer_norm_folding': False,
 'load_in_4bit': False,
 'model_name': 'gpt2',
 'n_ctx': 1024,
 'n_devices': 1,
 'n_heads': 12,
 'n_key_value_heads': None,


In [7]:
@dataclass
class Config:
    d_model: int = 768
    debug: bool = True
    layer_norm_eps: float = 1e-5
    d_vocab: int = 50257
    init_range: float = 0.02
    n_ctx: int = 1024
    d_head: int = 64
    d_mlp: int = 3072
    n_heads: int = 12
    n_layers: int = 12


cfg = Config()
print(cfg)

Config(d_model=768, debug=True, layer_norm_eps=1e-05, d_vocab=50257, init_range=0.02, n_ctx=1024, d_head=64, d_mlp=3072, n_heads=12, n_layers=12)


In [8]:
class LayerNorm(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.w = nn.Parameter(t.ones(cfg.d_model))
        self.b = nn.Parameter(t.zeros(cfg.d_model))

    def forward(self, residual: Float[Tensor, "batch posn d_model"]) -> Float[Tensor, "batch posn d_model"]:
        mean = residual.mean(dim=-1, keepdim=True)
        variance = residual.var(dim=-1, keepdim=True,correction=0)
        return ((residual-mean)/(t.sqrt(variance+self.cfg.layer_norm_eps))) * self.w + self.b


tests.rand_float_test(LayerNorm, [2, 4, 768])
tests.test_layer_norm_epsilon(LayerNorm, cache["resid_post", 11])
tests.load_gpt2_test(LayerNorm, reference_gpt2.ln_final, cache["resid_post", 11])

Input shape: torch.Size([2, 4, 768])
Output shape: torch.Size([2, 4, 768]) 

All tests in `test_layer_norm_epsilon` passed!
Input shape: torch.Size([1, 6, 768])
Output shape: torch.Size([1, 6, 768])
Reference output shape: torch.Size([1, 6, 768]) 

100.00% of the values are correct



In [9]:
class Embed(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.W_E = nn.Parameter(t.empty((cfg.d_vocab, cfg.d_model)))
        nn.init.normal_(self.W_E, std=self.cfg.init_range)

    def forward(self, tokens: Int[Tensor, "batch position"]) -> Float[Tensor, "batch position d_model"]:
        return self.W_E[tokens]

tests.test_embed(Embed)
tests.rand_int_test(Embed, [2, 4])
tests.load_gpt2_test(Embed, reference_gpt2.embed, tokens)

All tests in `test_embed` passed!
Input shape: torch.Size([2, 4])
Output shape: torch.Size([2, 4, 768]) 

Input shape: torch.Size([1, 6])
Output shape: torch.Size([1, 6, 768])
Reference output shape: torch.Size([1, 6, 768]) 

100.00% of the values are correct



In [10]:
class PosEmbed(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.W_pos = nn.Parameter(t.empty((cfg.n_ctx, cfg.d_model)))
        nn.init.normal_(self.W_pos, std=self.cfg.init_range)

    def forward(self, tokens: Int[Tensor, "batch position"]) -> Float[Tensor, "batch position d_model"]:
        # pos = t.tensor(list(range(len(tokens[-1])))).repeat(len(tokens),1)
        # return self.W_pos[pos]
        seq_lens = len(tokens[-1])
        batch = len(tokens)
        return einops.repeat(self.W_pos[:seq_lens], "input d_model -> batch input d_model", batch=batch)

tests.test_pos_embed(PosEmbed)
tests.rand_int_test(PosEmbed, [2, 4])
tests.load_gpt2_test(PosEmbed, reference_gpt2.pos_embed, tokens)

All tests in `test_pos_embed` passed!
Input shape: torch.Size([2, 4])
Output shape: torch.Size([2, 4, 768]) 

Input shape: torch.Size([1, 6])
Output shape: torch.Size([1, 6, 768])
Reference output shape: torch.Size([1, 6, 768]) 

100.00% of the values are correct



In [11]:
import circuitsvis as cv
from IPython.display import display

display(
    cv.attention.attention_patterns(
        tokens=reference_gpt2.to_str_tokens(reference_text), attention=cache["pattern", 0][0]
    )
)

In [12]:
display(
    cv.attention.attention_heads(
        tokens=reference_gpt2.to_str_tokens(reference_text), attention=cache["pattern", 0][0]
    )
)

In [13]:
class Attention(nn.Module):
    IGNORE: Float[Tensor, ""]

    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.W_Q = nn.Parameter(t.empty((cfg.n_heads, cfg.d_model, cfg.d_head)))
        self.W_K = nn.Parameter(t.empty((cfg.n_heads, cfg.d_model, cfg.d_head)))
        self.W_V = nn.Parameter(t.empty((cfg.n_heads, cfg.d_model, cfg.d_head)))
        self.W_O = nn.Parameter(t.empty((cfg.n_heads, cfg.d_head, cfg.d_model)))
        self.b_Q = nn.Parameter(t.zeros((cfg.n_heads, cfg.d_head)))
        self.b_K = nn.Parameter(t.zeros((cfg.n_heads, cfg.d_head)))
        self.b_V = nn.Parameter(t.zeros((cfg.n_heads, cfg.d_head)))
        self.b_O = nn.Parameter(t.zeros((cfg.d_model)))
        nn.init.normal_(self.W_Q, std=self.cfg.init_range)
        nn.init.normal_(self.W_K, std=self.cfg.init_range)
        nn.init.normal_(self.W_V, std=self.cfg.init_range)
        nn.init.normal_(self.W_O, std=self.cfg.init_range)
        self.register_buffer("IGNORE", t.tensor(float("-inf"), dtype=t.float32, device=device))

    def forward(self, normalized_resid_pre: Float[Tensor, "batch posn d_model"]) -> Float[Tensor, "batch posn d_model"]:
        # einops.einsum(
        #         normalized_resid_pre,
        #         self.W_Q,
        #         "batch posn d_model, nheads d_model d_head -> batch posn nheads d_head",
        #     )
        #     + self.b_Q
        Q = einops.einsum(
            normalized_resid_pre, self.W_Q,
            "batch posn d_model, n_head d_model d_head -> batch posn n_head d_head",
        ) + self.b_Q
        K = einops.einsum(
            normalized_resid_pre, self.W_K,
            "batch posn d_model, n_head d_model d_head -> batch posn n_head d_head",
        ) + self.b_K
        V = einops.einsum(
            normalized_resid_pre, self.W_V,
            "batch posn d_model, n_head d_model d_head -> batch posn n_head d_head",
        ) + self.b_V
        # t.sqrt(self.d_head)

        attn_scores = einops.einsum(
            Q, K,
            "batch posn_Q n_head d_head, batch posn_K n_head d_head -> batch n_head posn_Q posn_K",
        ) / self.cfg.d_head**0.5

        attn_pattern = self.apply_causal_mask(attn_scores).softmax(-1)
        
        Z = einops.einsum(
            attn_pattern, V,
            "batch n_head posn_Q posn_K, batch posn_K n_head d_head -> batch posn_Q n_head d_head"
        )

        return einops.einsum(
            Z, self.W_O,
            "batch posn_Q n_head d_head, n_head d_head d_model -> batch posn_Q d_model",
        ) + self.b_O

    def apply_causal_mask(
        self, attn_scores: Float[Tensor, "batch n_heads query_pos key_pos"]
    ) -> Float[Tensor, "batch n_heads query_pos key_pos"]:
        """
        Applies a causal mask to attention scores, and returns masked scores.
        """
        all_ones = t.ones(attn_scores.shape[-2], attn_scores.shape[-1], device=attn_scores.device)
        mask = t.triu(all_ones, diagonal=1).bool()
        # mask1 = t.triu(all_ones, diagonal=0).bool()
        # mask2 = t.triu(all_ones, diagonal=-1).bool()
        # print(mask, mask1, mask2)
        return attn_scores.masked_fill_(mask, self.IGNORE)


tests.test_causal_mask(Attention.apply_causal_mask)
tests.rand_float_test(Attention, [2, 4, 768])
tests.load_gpt2_test(Attention, reference_gpt2.blocks[0].attn, cache["normalized", 0, "ln1"])

All tests in `test_causal_mask` passed!
Input shape: torch.Size([2, 4, 768])
Output shape: torch.Size([2, 4, 768]) 

Input shape: torch.Size([1, 6, 768])
Output shape: torch.Size([1, 6, 768])
Reference output shape: torch.Size([1, 6, 768]) 

100.00% of the values are correct



In [14]:
class MLP(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.W_in = nn.Parameter(t.empty((cfg.d_model, cfg.d_mlp)))
        self.W_out = nn.Parameter(t.empty((cfg.d_mlp, cfg.d_model)))
        self.b_in = nn.Parameter(t.zeros((cfg.d_mlp)))
        self.b_out = nn.Parameter(t.zeros((cfg.d_model)))
        nn.init.normal_(self.W_in, std=self.cfg.init_range)
        nn.init.normal_(self.W_out, std=self.cfg.init_range)

    def forward(self, normalized_resid_mid: Float[Tensor, "batch posn d_model"]) -> Float[Tensor, "batch posn d_model"]:
        x_input = einops.einsum(
            normalized_resid_mid, self.W_in,
            "batch posn d_model, d_model d_mlp -> batch posn d_mlp",
        ) + self.b_in
        
        # this step makes the MLP operations non-linear
        x_input = gelu_new(x_input)
        
        x_output = einops.einsum(
            x_input, self.W_out,
            "batch posn d_mlp, d_mlp d_model -> batch posn d_model",
        ) + self.b_out
        return x_output

tests.rand_float_test(MLP, [2, 4, 768])
tests.load_gpt2_test(MLP, reference_gpt2.blocks[0].mlp, cache["normalized", 0, "ln2"])

Input shape: torch.Size([2, 4, 768])
Output shape: torch.Size([2, 4, 768]) 

Input shape: torch.Size([1, 6, 768])
Output shape: torch.Size([1, 6, 768])
Reference output shape: torch.Size([1, 6, 768]) 

100.00% of the values are correct



In [15]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.ln1 = LayerNorm(cfg)
        self.attn = Attention(cfg)
        self.ln2 = LayerNorm(cfg)
        self.mlp = MLP(cfg)

    def forward(self, resid_pre: Float[Tensor, "batch position d_model"]) -> Float[Tensor, "batch position d_model"]:
        attn = self.attn(self.ln1(resid_pre)) + resid_pre
        mlp = self.mlp(self.ln2(attn)) + attn
        return mlp


tests.rand_float_test(TransformerBlock, [2, 4, 768])
tests.load_gpt2_test(TransformerBlock, reference_gpt2.blocks[0], cache["resid_pre", 0])

Input shape: torch.Size([2, 4, 768])
Output shape: torch.Size([2, 4, 768]) 

Input shape: torch.Size([1, 6, 768])
Output shape: torch.Size([1, 6, 768])
Reference output shape: torch.Size([1, 6, 768]) 

100.00% of the values are correct



In [16]:
class Unembed(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.W_U = nn.Parameter(t.empty((cfg.d_model, cfg.d_vocab)))
        nn.init.normal_(self.W_U, std=self.cfg.init_range)
        self.b_U = nn.Parameter(t.zeros((cfg.d_vocab)), requires_grad=False)

    def forward(
        self, normalized_resid_final: Float[Tensor, "batch position d_model"]
    ) -> Float[Tensor, "batch position d_vocab"]:
        return einops.einsum(
            normalized_resid_final, self.W_U,
            "batch posn d_model, d_model d_vocab -> batch posn d_vocab",
        ) + self.b_U


tests.test_unembed(Unembed)
tests.rand_float_test(Unembed, [2, 4, 768])
tests.load_gpt2_test(Unembed, reference_gpt2.unembed, cache["ln_final.hook_normalized"])

All tests in `test_unembed` passed!
Input shape: torch.Size([2, 4, 768])
Output shape: torch.Size([2, 4, 50257]) 

Input shape: torch.Size([1, 6, 768])
Output shape: torch.Size([1, 6, 50257])
Reference output shape: torch.Size([1, 6, 50257]) 

100.00% of the values are correct



In [17]:
class DemoTransformer(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.embed = Embed(cfg)
        self.pos_embed = PosEmbed(cfg)
        self.blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.n_layers)])
        self.ln_final = LayerNorm(cfg)
        self.unembed = Unembed(cfg)

    def forward(self, tokens: Int[Tensor, "batch position"]) -> Float[Tensor, "batch position d_vocab"]:
        embed = self.embed(tokens)
        pos_embed = self.pos_embed(tokens)
        res = embed + pos_embed
        for block in self.blocks:
            res = block(res)
        res_ln = self.ln_final(res)
        logits = self.unembed(res_ln)
        return logits

tests.rand_int_test(DemoTransformer, [2, 4])
tests.load_gpt2_test(DemoTransformer, reference_gpt2, tokens)

Input shape: torch.Size([2, 4])
Output shape: torch.Size([2, 4, 50257]) 

Input shape: torch.Size([1, 6])
Output shape: torch.Size([1, 6, 50257])
Reference output shape: torch.Size([1, 6, 50257]) 

100.00% of the values are correct



In [18]:
demo_gpt2 = DemoTransformer(Config(debug=False)).to(device)
demo_gpt2.load_state_dict(reference_gpt2.state_dict(), strict=False)

demo_logits = demo_gpt2(tokens)

In [19]:
def get_log_probs(
    logits: Float[Tensor, "batch posn d_vocab"], tokens: Int[Tensor, "batch posn"]
) -> Float[Tensor, "batch posn-1"]:
    log_probs = logits.log_softmax(dim=-1)
    # Get logprobs for the first seq_len-1 predictions (so we can compare them with the actual next tokens)
    log_probs_for_tokens = log_probs[:, :-1].gather(dim=-1, index=tokens[:, 1:].unsqueeze(-1)).squeeze(-1)

    return log_probs_for_tokens


pred_log_probs = get_log_probs(demo_logits, tokens)
print(f"Avg cross entropy loss: {-pred_log_probs.mean():.4f}")
print(f"Avg cross entropy loss for uniform distribution: {math.log(demo_gpt2.cfg.d_vocab):4f}")
print(f"Avg probability assigned to correct token: {pred_log_probs.exp().mean():4f}")

Avg cross entropy loss: 4.3414
Avg cross entropy loss for uniform distribution: 10.824905
Avg probability assigned to correct token: 0.136537


In [20]:
test_string = """Mitigating the risk of extinction from AI should be a global priority alongside other societal-scale risks such as"""
for i in tqdm(range(100)):
    test_tokens = reference_gpt2.to_tokens(test_string).to(device)
    demo_logits = demo_gpt2(test_tokens)
    test_string += reference_gpt2.tokenizer.decode(demo_logits[-1, -1].argmax())

print(test_string)

  0%|          | 0/100 [00:00<?, ?it/s]

Mitigating the risk of extinction from AI should be a global priority alongside other societal-scale risks such as climate change, the spread of infectious diseases and the spread of infectious diseases.


The research is published in the journal Nature Communications.


The research team is led by Dr. Michael J. H. Haldane, a professor of biology at the University of California, Berkeley, and co-author of the paper.


"We are very excited to see that the AI community is starting to take notice of the potential for AI to be a major threat to the human race,"
